# Notebook 01 — CICIDS-2017 Data Exploration

Explores the CICIDS-2017 dataset used for LSTM attack prediction in CM-MTD.

**Sections:**
1. Load dataset (real or synthetic)
2. Class distribution
3. Feature statistics
4. Temporal attack patterns
5. Event sequence visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import compat
from config import load_config

cfg = load_config('../config/config.yaml')
plt.rcParams['figure.dpi'] = 120
print('Config loaded:', cfg['experiment']['name'])

## 1. Load Data

In [ ]:
from datasets.sequence_builder import SyntheticDataGenerator
from datasets.cicids2017_loader import CLASS_NAMES, ATTACK_CLASS_MAP

# Try real CICIDS2017 first, fall back to synthetic
try:
    from datasets.cicids2017_loader import CICIDS2017Loader
    from datasets.preprocessor import CICIDS2017Preprocessor
    loader = CICIDS2017Loader(data_dir='../data/cicids2017')
    df_raw = loader.load_all()
    pre = CICIDS2017Preprocessor(config=cfg)
    X, y, feat_names = pre.fit_transform(df_raw)
    data_source = 'CICIDS-2017 (real)'
    print(f'Loaded {len(X):,} flows from {data_source}')
except Exception as e:
    print(f'Real data not found ({e}) — using synthetic')
    gen = SyntheticDataGenerator(n_classes=8, seed=42)
    X, y = gen.generate(n_samples=50_000)
    feat_names = [f'feat_{i}' for i in range(X.shape[1])]
    data_source = 'Synthetic (CICIDS-2017 distribution)'

print(f'X: {X.shape}  y: {y.shape}')
print(f'Classes: {dict(zip(CLASS_NAMES, np.bincount(y, minlength=8)))}')

## 2. Class Distribution

In [ ]:
counts = np.bincount(y, minlength=8)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
colors = plt.cm.tab10(np.linspace(0, 1, 8))
axes[0].bar(CLASS_NAMES, counts, color=colors, edgecolor='white')
axes[0].set_xlabel('Attack Class')
axes[0].set_ylabel('Flow Count')
axes[0].set_title(f'Class Distribution — {data_source}')
axes[0].tick_params(axis='x', rotation=30)
for i, c in enumerate(counts):
    axes[0].text(i, c + counts.max()*0.01, f'{c:,}', ha='center', fontsize=8)

# Pie chart
non_zero = counts > 0
axes[1].pie(counts[non_zero], labels=np.array(CLASS_NAMES)[non_zero],
            colors=colors[non_zero], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proportions')

plt.tight_layout()
plt.savefig('../results/figures/png/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class imbalance ratio: {counts.max()/max(counts[counts>0].min(),1):.1f}x')

## 3. Feature Statistics

In [ ]:
# Per-class feature means (top 10 most discriminative features)
from sklearn.feature_selection import f_classif
F_stats, p_vals = f_classif(X, y)
top10_idx = np.argsort(F_stats)[::-1][:10]
top10_names = [feat_names[i] for i in top10_idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(range(10), F_stats[top10_idx], color='steelblue')
ax.set_yticks(range(10))
ax.set_yticklabels(top10_names, fontsize=9)
ax.set_xlabel('F-statistic (ANOVA)')
ax.set_title('Top 10 Discriminative Features (ANOVA F-score)')
plt.tight_layout()
plt.savefig('../results/figures/png/eda_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Event Sequence Patterns

In [ ]:
from datasets.sequence_builder import SecurityEventSequenceBuilder

# Build event sequences
builder = SecurityEventSequenceBuilder(sequence_length=10, n_classes=8)
X_seq, y_seq = builder.build_from_labels(y[:10000])
print(f'Sequences: X:{X_seq.shape}  y:{y_seq.shape}')

# Transition matrix: P(next_class | current_class)
trans = np.zeros((8, 8), dtype=float)
for i in range(len(y) - 1):
    trans[y[i], y[i+1]] += 1
row_sums = trans.sum(axis=1, keepdims=True)
trans_norm = np.where(row_sums > 0, trans / row_sums, 0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(trans_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Next Attack Class')
ax.set_ylabel('Current Attack Class')
ax.set_title('Attack Transition Matrix P(next | current)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/png/eda_transition_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Temporal Attack Timeline

In [ ]:
# Show first 500 events as a colored timeline
n_show = 500
colors_map = plt.cm.tab10(np.linspace(0, 1, 8))
fig, ax = plt.subplots(figsize=(14, 2.5))

for t in range(min(n_show, len(y))):
    ax.axvspan(t, t+1, color=colors_map[y[t]], alpha=0.9)

# Legend
from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=colors_map[i], label=CLASS_NAMES[i]) for i in range(8)]
ax.legend(handles=legend_elems, loc='upper right', ncol=4, fontsize=8)
ax.set_xlim(0, n_show)
ax.set_xlabel('Time Step')
ax.set_yticks([])
ax.set_title(f'Attack Timeline — First {n_show} events')
plt.tight_layout()
plt.savefig('../results/figures/png/eda_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Notebook complete. Figures saved to results/figures/png/')